# LLT batch C probe A: reward models, DPO, tool-use SFT

Writes `llt_c_probe_a.json`.

In [ ]:
TAG = 'a'

import json, os, subprocess, sys, time, platform
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.22.1", "trl==1.17.1", "peft==0.21.2", "accelerate==1.15.0",
                "bitsandbytes==0.50.2", "datasets==5.1.0"], check=False)
WORKER = '"""Batch C probe worker. One job per process. Usage: python probe_c_worker.py <job.json> <result.json>\njob.kind in: rm | dpo | grpo | toolsft | kd | kd_onpolicy | trl_rm | trl_dpo | trl_grpo | trl_distill\n"""\nimport gc, json, os, sys, time, traceback, statistics, io, logging\nos.environ["CUDA_VISIBLE_DEVICES"] = "0"\nos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"\njob = json.load(open(sys.argv[1])); outp = sys.argv[2]\nR = {"tag": job["tag"], "kind": job["kind"], "model": job.get("model")}\ndef save():\n    json.dump(R, open(outp, "w"), indent=1, default=str)\nimport torch\nimport transformers\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nlogbuf = io.StringIO()\n_h = logging.StreamHandler(logbuf); _h.setLevel(logging.WARNING)\nlogging.getLogger("transformers").addHandler(_h)\nDEV = "cuda:0"\n\ndef peak():\n    return round(torch.cuda.max_memory_allocated() / 1e9, 2)\ndef med(xs, skip=2):\n    xs = xs[skip:] if len(xs) > skip + 1 else xs\n    return round(statistics.median(xs), 3) if xs else None\ndef free():\n    gc.collect(); torch.cuda.empty_cache()\n\ndef load_tok(name):\n    tok = AutoTokenizer.from_pretrained(name)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\ndef lora_model(model, task="CAUSAL_LM", r=16):\n    from peft import LoraConfig, get_peft_model\n    names = [n for n, _ in model.named_modules()]\n    ex = r".*(visual|vision).*" if any(("visual" in n or "vision" in n) for n in names) else None\n    cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.0, target_modules="all-linear", exclude_modules=ex, task_type=task)\n    m = get_peft_model(model, cfg)\n    for p in m.parameters():\n        if p.requires_grad and p.dtype != torch.float32:\n            p.data = p.data.float()\n    return m\n\ndef prefs(tok, n, max_len):\n    """UltraFeedback binarized pairs -> list of (prompt_ids, chosen_ids, rejected_ids) with total length <= max_len."""\n    from datasets import load_dataset\n    ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split=f"train_prefs[:{n * 3}]")\n    out = []\n    for ex in ds:\n        p = tok(ex["prompt"], add_special_tokens=False).input_ids[:200]\n        c = tok("\\n\\n" + ex["chosen"][-1]["content"], add_special_tokens=False).input_ids\n        r = tok("\\n\\n" + ex["rejected"][-1]["content"], add_special_tokens=False).input_ids\n        c = c[: max_len - len(p)]; r = r[: max_len - len(p)]\n        if len(c) < 30 or len(r) < 30:\n            continue\n        out.append((p, c, r))\n        if len(out) >= n:\n            break\n    return out\n\ndef pad_batch(seqs, pad, L):\n    ids = torch.full((len(seqs), L), pad, dtype=torch.long); att = torch.zeros((len(seqs), L), dtype=torch.long)\n    for i, s in enumerate(seqs):\n        s = s[:L]; ids[i, :len(s)] = torch.tensor(s); att[i, :len(s)] = 1\n    return ids, att\n\ndef seq_logp(model, ids, att, plen):\n    """sum of token log-probs over the response part (positions >= plen) for each row. Computes log-softmax in chunks of rows."""\n    with torch.autocast("cuda", dtype=torch.float16):\n        logits = model(input_ids=ids, attention_mask=att).logits\n    lp = torch.log_softmax(logits[:, :-1].float(), dim=-1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)\n    pos = torch.arange(lp.shape[1], device=lp.device).unsqueeze(0)\n    mask = (pos >= (plen.unsqueeze(1) - 1)) & att[:, 1:].bool()\n    return (lp * mask).sum(-1), mask.sum(-1)\n\ndef timed(fn):\n    torch.cuda.synchronize(); t = time.time(); r = fn(); torch.cuda.synchronize(); return r, time.time() - t\n\ntry:\n    kind = job["kind"]; name = job.get("model")\n    T = job.get("seq", 512)\n    if kind == "rm":\n        from transformers import AutoModelForSequenceClassification\n        tok = load_tok(name)\n        data = prefs(tok, job["steps"] * job["pairs"], T)\n        t0 = time.time()\n        model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=1, dtype=torch.float32, device_map=DEV)\n        model.config.pad_token_id = tok.pad_token_id\n        R["loadedClass"] = type(model).__name__; R["loadSec"] = round(time.time() - t0, 1)\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False}); model.enable_input_require_grads()\n        model.config.use_cache = False\n        if job["mode"] == "lora":\n            model = lora_model(model, "SEQ_CLS")\n            opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)\n        else:\n            import bitsandbytes as bnb\n            opt = bnb.optim.AdamW8bit([p for p in model.parameters() if p.requires_grad], lr=1e-5)\n        tr = [p for p in model.parameters() if p.requires_grad]\n        R["trainableParams"] = sum(p.numel() for p in tr)\n        model.train(); scaler = torch.amp.GradScaler("cuda", init_scale=2048)\n        torch.cuda.reset_peak_memory_stats(); times = []; losses = []; accs = []; micro = job["pairs"]\n        for s in range(job["steps"]):\n            chunk = data[s * job["pairs"]:(s + 1) * job["pairs"]]\n            t1 = time.time()\n            while True:\n                try:\n                    opt.zero_grad(set_to_none=True); tl = 0; ta = 0\n                    for i in range(0, len(chunk), micro):\n                        sub = chunk[i:i + micro]\n                        seqs = [p + c for p, c, r in sub] + [p + r for p, c, r in sub]\n                        ids, att = pad_batch(seqs, tok.pad_token_id, T); ids = ids.cuda(); att = att.cuda()\n                        with torch.autocast("cuda", dtype=torch.float16):\n                            rew = model(input_ids=ids, attention_mask=att).logits.squeeze(-1).float()\n                        rc, rr = rew[:len(sub)], rew[len(sub):]\n                        loss = -torch.nn.functional.logsigmoid(rc - rr).mean() * (len(sub) / len(chunk))\n                        scaler.scale(loss).backward(); tl += loss.item(); ta += (rc > rr).float().sum().item()\n                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update()\n                    break\n                except torch.OutOfMemoryError:\n                    R.setdefault("oomRetries", []).append({"step": s, "micro": micro}); opt.zero_grad(set_to_none=True); free()\n                    if micro == 1: raise\n                    micro = max(1, micro // 2)\n            torch.cuda.synchronize(); times.append(time.time() - t1); losses.append(tl); accs.append(ta / len(chunk))\n            print(f"step {s+1} loss {tl:.4f} acc {accs[-1]:.2f} {times[-1]:.2f}s", flush=True)\n        R.update(stepsDone=len(times), pairsPerStep=job["pairs"], microPairs=micro, seq=T, sPerStepMedian=med(times), peakMemGB=peak(),\n                 seqPerSec=round(2 * job["pairs"] / med(times), 2), loss1=losses[0], lossLast=losses[-1], accs=[round(a, 2) for a in accs],\n                 stepTimes=[round(t, 2) for t in times], gradScalerFinal=scaler.get_scale(), ok=True)\n\n    elif kind == "dpo":\n        tok = load_tok(name)\n        data = prefs(tok, job["steps"] * job["pairs"], T)\n        t0 = time.time()\n        model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32, device_map=DEV)\n        R["loadedClass"] = type(model).__name__; R["loadSec"] = round(time.time() - t0, 1)\n        ref = None\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False}); model.enable_input_require_grads()\n        model.config.use_cache = False\n        if job["mode"] == "lora":\n            model = lora_model(model)\n            opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-5)\n        else:\n            import bitsandbytes as bnb\n            ref = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, device_map=DEV).eval()\n            for p in ref.parameters(): p.requires_grad = False\n            opt = bnb.optim.AdamW8bit([p for p in model.parameters() if p.requires_grad], lr=5e-7)\n        tr = [p for p in model.parameters() if p.requires_grad]\n        R["trainableParams"] = sum(p.numel() for p in tr)\n        model.train(); scaler = torch.amp.GradScaler("cuda", init_scale=2048)\n        torch.cuda.reset_peak_memory_stats(); times = []; refT = []; losses = []; margins = []; beta = 0.1\n        for s in range(job["steps"]):\n            chunk = data[s * job["pairs"]:(s + 1) * job["pairs"]]\n            opt.zero_grad(set_to_none=True); t1 = time.time(); tl = 0; tref = 0; mg = 0\n            for (p, c, r) in chunk:  # one pair per micro-batch (2 rows)\n                ids, att = pad_batch([p + c, p + r], tok.pad_token_id, len(p) + max(len(c), len(r))); ids = ids.cuda(); att = att.cuda()\n                plen = torch.tensor([len(p)] * 2, device=DEV)\n                def refpass():\n                    with torch.no_grad():\n                        if ref is not None:\n                            return seq_logp(ref, ids, att, plen)[0]\n                        with model.disable_adapter():\n                            model.eval(); x = seq_logp(model, ids, att, plen)[0]; model.train(); return x\n                rl, dtt = timed(refpass); tref += dtt\n                pl, _ = seq_logp(model, ids, att, plen)\n                z = beta * ((pl[0] - rl[0]) - (pl[1] - rl[1]))\n                loss = -torch.nn.functional.logsigmoid(z) / len(chunk)\n                scaler.scale(loss).backward(); tl += loss.item(); mg += z.item() / beta / len(chunk)\n            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update()\n            torch.cuda.synchronize(); times.append(time.time() - t1); refT.append(tref); losses.append(tl); margins.append(mg)\n            print(f"step {s+1} loss {tl:.4f} margin {mg:.3f} {times[-1]:.2f}s ref {tref:.2f}s", flush=True)\n        R.update(stepsDone=len(times), pairsPerStep=job["pairs"], seq=T, sPerStepMedian=med(times), refForwardSecMedian=med(refT), peakMemGB=peak(),\n                 loss1=losses[0], lossLast=losses[-1], marginLast=margins[-1], stepTimes=[round(t, 2) for t in times], ok=True,\n                 refMode="separate fp16 copy" if ref is not None else "adapter disabled (no extra copy)")\n\n    elif kind == "toolsft":\n        from datasets import load_dataset\n        tok = load_tok(name)\n        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")\n        text = "\\n".join(t for t in ds["text"][:20000] if len(t.strip()) > 40)[:900000]\n        ids_all = tok(text, return_tensors="pt").input_ids[0]\n        B = job["batch"]; N = job["steps"]\n        stream = ids_all[: B * T * N].view(N, B, T)\n        t0 = time.time()\n        if job["mode"] == "qlora":\n            q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\n            model = AutoModelForCausalLM.from_pretrained(name, quantization_config=q, dtype=torch.float16, device_map=DEV)\n        else:\n            model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, device_map=DEV)\n        R["loadedClass"] = type(model).__name__; R["loadSec"] = round(time.time() - t0, 1)\n        for p in model.parameters(): p.requires_grad = False\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False}); model.enable_input_require_grads()\n        model.config.use_cache = False\n        model = lora_model(model)\n        tr = [p for p in model.parameters() if p.requires_grad]\n        R["trainableParams"] = sum(p.numel() for p in tr)\n        opt = torch.optim.AdamW(tr, lr=2e-4); model.train(); scaler = torch.amp.GradScaler("cuda", init_scale=2048)\n        torch.cuda.reset_peak_memory_stats(); times = []; losses = []; micro = job.get("micro", B); nonf = 0\n        for s in range(N):\n            t1 = time.time()\n            while True:\n                try:\n                    opt.zero_grad(set_to_none=True); tl = 0\n                    for i in range(0, B, micro):\n                        x = stream[s][i:i + micro].cuda()\n                        with torch.autocast("cuda", dtype=torch.float16):\n                            loss = model(input_ids=x, labels=x).loss\n                        frac = x.shape[0] / B; tl += loss.item() * frac; scaler.scale(loss * frac).backward()\n                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update()\n                    break\n                except torch.OutOfMemoryError:\n                    R.setdefault("oomRetries", []).append({"step": s, "micro": micro}); opt.zero_grad(set_to_none=True); free()\n                    if micro == 1: raise\n                    micro //= 2\n            torch.cuda.synchronize(); times.append(time.time() - t1); losses.append(tl)\n            nonf += int(not (tl == tl and abs(tl) != float("inf")))\n            print(f"step {s+1} loss {tl:.4f} {times[-1]:.2f}s", flush=True)\n        R.update(stepsDone=len(times), batch=B, seq=T, microBatch=micro, sPerStepMedian=med(times), tokensPerSec=round(B * T / med(times), 1), peakMemGB=peak(),\n                 loss1=losses[0], lossLast=losses[-1], nonFinite=nonf, stepTimes=[round(t, 2) for t in times], ok=True)\n\n    elif kind == "grpo":\n        from datasets import load_dataset\n        tok = load_tok(name); tok.padding_side = "left"\n        ds = load_dataset("openai/gsm8k", "main", split="train[:64]")\n        prompts = [f"Question: {q}\\nSolve step by step, then give the final answer after \'####\'.\\nAnswer:" for q in ds["question"]]\n        t0 = time.time()\n        model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, device_map=DEV)\n        R["loadedClass"] = type(model).__name__; R["loadSec"] = round(time.time() - t0, 1)\n        for p in model.parameters(): p.requires_grad = False\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False}); model.enable_input_require_grads()\n        model = lora_model(model)\n        tr = [p for p in model.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(tr, lr=1e-5); scaler = torch.amp.GradScaler("cuda", init_scale=2048)\n        R["trainableParams"] = sum(p.numel() for p in tr)\n        NEW = job.get("new", 256); rows = []\n        torch.cuda.reset_peak_memory_stats()\n        for it in range(job["iters"]):\n            for (nprompt, G) in job["shapes"]:\n                batch_prompts = [pr for pr in prompts[it * nprompt:(it + 1) * nprompt] for _ in range(G)]\n                enc = tok(batch_prompts, return_tensors="pt", padding=True).to(DEV)\n                model.eval(); model.config.use_cache = True; torch.cuda.reset_peak_memory_stats()\n                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):\n                    out, tg = timed(lambda: model.generate(**enc, max_new_tokens=NEW, do_sample=True, temperature=1.0, top_p=1.0, top_k=0, pad_token_id=tok.pad_token_id))\n                gen = out[:, enc.input_ids.shape[1]:]\n                eos = tok.eos_token_id\n                lens = []\n                for row in gen.tolist():\n                    n = len(row)\n                    for j, t in enumerate(row):\n                        if t == eos or t == tok.pad_token_id: n = j + 1; break\n                    lens.append(n)\n                genpeak = peak()\n                # policy-gradient update on these samples (random advantages: only timing and memory are measured)\n                model.train(); model.config.use_cache = False; torch.cuda.reset_peak_memory_stats()\n                adv = torch.randn(len(lens), device=DEV)\n                full_att = torch.cat([enc.attention_mask, torch.ones_like(gen)], 1)\n                def upd():\n                    opt.zero_grad(set_to_none=True); mb = job.get("micro", 4)\n                    for i in range(0, out.shape[0], mb):\n                        o = out[i:i + mb]; a = full_att[i:i + mb]\n                        with torch.autocast("cuda", dtype=torch.float16):\n                            lg = model(input_ids=o, attention_mask=a).logits[:, enc.input_ids.shape[1] - 1:-1]\n                        lp = torch.log_softmax(lg.float(), -1).gather(-1, gen[i:i + mb].unsqueeze(-1)).squeeze(-1)\n                        m = torch.zeros_like(lp)\n                        for k, n in enumerate(lens[i:i + mb]): m[k, :n] = 1\n                        loss = -((lp * adv[i:i + mb].unsqueeze(1) * m).sum() / m.sum()) * (o.shape[0] / out.shape[0])\n                        scaler.scale(loss).backward()\n                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update()\n                _, tu = timed(upd)\n                rows.append(dict(iter=it, prompts=nprompt, G=G, completions=len(lens), promptTokens=int(enc.input_ids.shape[1]), genSec=round(tg, 2),\n                                 genTokens=int(sum(lens)), genTokPerSec=round(sum(lens) / tg, 1), meanLen=round(sum(lens) / len(lens), 1),\n                                 hitCap=int(sum(1 for n in lens if n >= NEW)), updSec=round(tu, 2), genPeakGB=genpeak, updPeakGB=peak()))\n                print(rows[-1], flush=True); R["rows"] = rows; save()\n                free()\n        R.update(ok=True)\n        R["sample"] = tok.decode(gen[0][:lens[0]], skip_special_tokens=True)[:300]\n\n    elif kind in ("kd", "kd_onpolicy"):\n        from datasets import load_dataset\n        tname = job["teacher"]; sname = name\n        tok = load_tok(sname)\n        ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs[:400]")\n        texts = [e["prompt"] + "\\n\\n" + e["chosen"][-1]["content"] for e in ds]\n        t0 = time.time()\n        if job["teacher_quant"]:\n            q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\n            teacher = AutoModelForCausalLM.from_pretrained(tname, quantization_config=q, dtype=torch.float16, device_map=DEV)\n        else:\n            teacher = AutoModelForCausalLM.from_pretrained(tname, dtype=torch.float16, device_map=DEV)\n        teacher.eval()\n        for p in teacher.parameters(): p.requires_grad = False\n        R["teacherLoadSec"] = round(time.time() - t0, 1); R["teacherGB"] = round(torch.cuda.memory_allocated() / 1e9, 2)\n        R["teacherVocab"] = teacher.config.get_text_config().vocab_size\n        t0 = time.time()\n        if job["student_mode"] == "full":\n            student = AutoModelForCausalLM.from_pretrained(sname, dtype=torch.float32, device_map=DEV)\n            import bitsandbytes as bnb\n            student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False}); student.enable_input_require_grads()\n            opt = bnb.optim.AdamW8bit([p for p in student.parameters() if p.requires_grad], lr=1e-5)\n        else:\n            student = AutoModelForCausalLM.from_pretrained(sname, dtype=torch.float16, device_map=DEV)\n            for p in student.parameters(): p.requires_grad = False\n            student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False}); student.enable_input_require_grads()\n            student = lora_model(student)\n            opt = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad], lr=1e-4)\n        student.config.use_cache = False\n        tr = [p for p in student.parameters() if p.requires_grad]\n        R["studentLoadSec"] = round(time.time() - t0, 1); R["studentVocab"] = student.get_base_model().config.get_text_config().vocab_size if hasattr(student, "get_base_model") else student.config.get_text_config().vocab_size\n        R["trainableParams"] = sum(p.numel() for p in tr); R["residentGBBeforeStep"] = round(torch.cuda.memory_allocated() / 1e9, 2)\n        scaler = torch.amp.GradScaler("cuda", init_scale=2048); torch.cuda.reset_peak_memory_stats()\n        B = job["batch"]; rows = []; times = []\n        tok.padding_side = "right"\n        for s in range(job["steps"]):\n            t1 = time.time(); rec = {}\n            if kind == "kd":\n                enc = tok(texts[s * B:(s + 1) * B], return_tensors="pt", padding="max_length", truncation=True, max_length=T).to(DEV)\n                ids, att = enc.input_ids, enc.attention_mask\n                micro = job.get("micro", B); opt.zero_grad(set_to_none=True); tl = 0\n                for i in range(0, B, micro):\n                    x = ids[i:i + micro]; a = att[i:i + micro]\n                    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):\n                        tl_logits = teacher(input_ids=x, attention_mask=a).logits.float()\n                    with torch.autocast("cuda", dtype=torch.float16):\n                        sl = student(input_ids=x, attention_mask=a).logits.float()\n                    m = a[:, 1:].float()\n                    # forward KL(teacher || student) per token\n                    tp = torch.log_softmax(tl_logits[:, :-1], -1); sp = torch.log_softmax(sl[:, :-1], -1)\n                    kl = (tp.exp() * (tp - sp)).sum(-1)\n                    loss = (kl * m).sum() / m.sum() * (x.shape[0] / B)\n                    scaler.scale(loss).backward(); tl += loss.item()\n                    del tl_logits, sl, tp, sp, kl\n                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update()\n                rec = dict(loss=tl)\n            else:\n                tok.padding_side = "left"\n                pr = [e["prompt"][:400] for e in ds.select(range(s * B, (s + 1) * B))]\n                enc = tok(pr, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEV)\n                student.eval(); student.config.use_cache = True\n                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):\n                    out, tg = timed(lambda: student.generate(**enc, max_new_tokens=job["new"], do_sample=True, temperature=1.0, top_k=0, pad_token_id=tok.pad_token_id))\n                student.train(); student.config.use_cache = False\n                P = enc.input_ids.shape[1]; gen = out[:, P:]\n                att = torch.cat([enc.attention_mask, torch.ones_like(gen)], 1)\n                def tscore():\n                    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):\n                        lg = teacher(input_ids=out, attention_mask=att).logits[:, P - 1:-1].float()\n                        return torch.log_softmax(lg, -1).gather(-1, gen.unsqueeze(-1)).squeeze(-1)\n                tlp, tt = timed(tscore)\n                def sup():\n                    opt.zero_grad(set_to_none=True)\n                    with torch.autocast("cuda", dtype=torch.float16):\n                        lg = student(input_ids=out, attention_mask=att).logits[:, P - 1:-1]\n                    slp = torch.log_softmax(lg.float(), -1).gather(-1, gen.unsqueeze(-1)).squeeze(-1)\n                    # per-token reverse-KL "reward" (teacher logp - student logp) as the advantage, REINFORCE-style\n                    adv = (tlp - slp).detach()\n                    loss = -(slp * adv).mean()\n                    scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update()\n                    return float(adv.mean())\n                am, tu = timed(sup)\n                rec = dict(genSec=round(tg, 2), teacherScoreSec=round(tt, 2), updSec=round(tu, 2), meanRevKLEstimate=-am, genTokens=int(gen.numel()))\n            torch.cuda.synchronize(); times.append(time.time() - t1); rec["stepSec"] = round(times[-1], 2); rows.append(rec); print(rec, flush=True)\n        R.update(rows=rows, batch=B, seq=T, sPerStepMedian=med(times, 1), peakMemGB=peak(), ok=True)\n\n    elif kind.startswith("trl_"):\n        import trl\n        from datasets import load_dataset\n        from peft import LoraConfig\n        R["trlVersion"] = trl.__version__\n        tok = load_tok(name)\n        lc = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM" if kind != "trl_rm" else "SEQ_CLS")\n        mk = dict(dtype=torch.float32)\n        t0 = time.time()\n        if kind == "trl_rm":\n            from trl import RewardTrainer, RewardConfig\n            ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:96]")\n            cfg = RewardConfig(output_dir="/kaggle/working/o_rm", max_length=512, per_device_train_batch_size=2, gradient_accumulation_steps=2, max_steps=job["steps"],\n                               fp16=True, gradient_checkpointing=True, report_to="none", logging_steps=1, learning_rate=1e-4, save_strategy="no", model_init_kwargs=mk)\n            trainer = RewardTrainer(model=name, args=cfg, train_dataset=ds, processing_class=tok, peft_config=lc)\n        elif kind == "trl_dpo":\n            from trl import DPOTrainer, DPOConfig\n            ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:96]")\n            cfg = DPOConfig(output_dir="/kaggle/working/o_dpo", max_length=512, per_device_train_batch_size=2, gradient_accumulation_steps=2, max_steps=job["steps"],\n                            fp16=True, gradient_checkpointing=True, report_to="none", logging_steps=1, learning_rate=5e-6, save_strategy="no", model_init_kwargs=mk)\n            trainer = DPOTrainer(model=name, args=cfg, train_dataset=ds, processing_class=tok, peft_config=lc)\n        elif kind == "trl_grpo":\n            from trl import GRPOTrainer, GRPOConfig\n            g = load_dataset("openai/gsm8k", "main", split="train[:64]")\n            ds = g.map(lambda e: {"prompt": f"Question: {e[\'question\']}\\nSolve step by step, then give the final answer after \'####\'.\\nAnswer:", "gt": e["answer"].split("####")[-1].strip()})\n            def reward(completions, gt, **kw):\n                return [1.0 if str(a) in c else 0.0 for c, a in zip(completions, gt)]\n            cfg = GRPOConfig(output_dir="/kaggle/working/o_grpo", num_generations=4, per_device_train_batch_size=4, gradient_accumulation_steps=1, max_completion_length=job.get("new", 128),\n                             max_steps=job["steps"], fp16=True, gradient_checkpointing=True, report_to="none", logging_steps=1, learning_rate=1e-5, save_strategy="no", model_init_kwargs=mk,\n                             use_vllm=False, beta=0.0)\n            trainer = GRPOTrainer(model=name, reward_funcs=reward, args=cfg, train_dataset=ds, processing_class=tok, peft_config=lc)\n        elif kind == "trl_distill":\n            from trl import DistillationTrainer, DistillationConfig\n            ds = load_dataset("trl-lib/tldr", split="train[:64]") if False else load_dataset("openai/gsm8k", "main", split="train[:64]").map(lambda e: {"prompt": f"Question: {e[\'question\']}\\nAnswer:"}, remove_columns=["question", "answer"])\n            cfg = DistillationConfig(output_dir="/kaggle/working/o_dist", per_device_train_batch_size=4, gradient_accumulation_steps=1, max_completion_length=job.get("new", 128),\n                                     max_steps=job["steps"], fp16=True, gradient_checkpointing=True, report_to="none", logging_steps=1, learning_rate=1e-5, save_strategy="no",\n                                     model_init_kwargs=mk, teacher_model_init_kwargs=dict(dtype=torch.float16), beta=1.0)\n            trainer = DistillationTrainer(model=name, teacher_model=job["teacher"], args=cfg, train_dataset=ds, processing_class=tok, peft_config=lc)\n        R["initSec"] = round(time.time() - t0, 1)\n        # trap check from batch B: TRL may recast LoRA weights to bf16\n        dts = sorted({str(p.dtype) for p in trainer.model.parameters() if p.requires_grad}); R["trainableDtypesAfterInit"] = dts\n        for p in trainer.model.parameters():\n            if p.requires_grad and p.dtype != torch.float32: p.data = p.data.float()\n        torch.cuda.reset_peak_memory_stats(); t1 = time.time()\n        trainer.train()\n        torch.cuda.synchronize(); R["trainWallSec"] = round(time.time() - t1, 1); R["sPerStepMean"] = round(R["trainWallSec"] / job["steps"], 2); R["peakMemGB"] = peak()\n        R["log"] = [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in h.items()} for h in trainer.state.log_history][-job["steps"] - 1:]\n        R["ok"] = True\n    R["warnings"] = logbuf.getvalue()[:1500]\nexcept Exception as e:\n    R["ok"] = False; R["error"] = f"{type(e).__name__}: {str(e)[:900]}"; R["trace"] = traceback.format_exc()[-2200:]\n    R["warnings"] = logbuf.getvalue()[:1500]\n    try: R["peakMemGB"] = peak()\n    except Exception: pass\nsave()\n'
open("/kaggle/working/probe_c_worker.py", "w").write(WORKER)
import torch, transformers, peft, trl, accelerate, bitsandbytes, datasets
ALL = {"date": time.strftime("%Y-%m-%d"), "gpu": torch.cuda.get_device_name(0), "gpuCount": torch.cuda.device_count(),
       "env": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda, "transformers": transformers.__version__,
               "trl": trl.__version__, "peft": peft.__version__, "accelerate": accelerate.__version__, "bitsandbytes": bitsandbytes.__version__, "datasets": datasets.__version__},
       "runs": []}
ALL["trlSurface"] = {n: hasattr(trl, n) for n in ["PPOTrainer", "RewardTrainer", "DPOTrainer", "KTOTrainer", "GRPOTrainer", "RLOOTrainer", "DistillationTrainer", "SFTTrainer", "ORPOTrainer", "CPOTrainer", "GKDTrainer"]}
import importlib
for mod, cls in [("trl.experimental.orpo", "ORPOTrainer"), ("trl.experimental.cpo", "CPOTrainer"), ("trl.experimental.gkd", "GKDTrainer"), ("trl.experimental.gold", "GOLDTrainer"), ("trl.experimental.online_dpo", "OnlineDPOTrainer")]:
    try:
        ALL["trlSurface"][mod + "." + cls] = hasattr(importlib.import_module(mod), cls)
    except Exception as e:
        ALL["trlSurface"][mod + "." + cls] = "ERR " + type(e).__name__
print(ALL, flush=True)
T0 = time.time()
def save():
    json.dump(ALL, open("/kaggle/working/llt_c_probe_%s.json" % TAG, "w"), indent=1)
def job(tag, kind, model, cap=420, **kw):
    jf = "/kaggle/working/job_%s.json" % tag; rf = "/kaggle/working/res_%s.json" % tag
    json.dump(dict(tag=tag, kind=kind, model=model, **kw), open(jf, "w"))
    t = time.time()
    try:
        p = subprocess.run([sys.executable, "/kaggle/working/probe_c_worker.py", jf, rf], capture_output=True, text=True, timeout=cap)
        tail = (p.stdout or "")[-1200:] + (p.stderr or "")[-1200:]
    except subprocess.TimeoutExpired as e:
        tail = "TIMEOUT after %ss" % cap
    try:
        r = json.load(open(rf))
    except Exception:
        r = {"tag": tag, "ok": False, "error": "no result file / crashed / timeout"}
    r["wallSec"] = round(time.time() - t, 1); r["tail"] = tail
    ALL["runs"].append(r); save()
    print(tag, {k: r.get(k) for k in ("ok", "sPerStepMedian", "sPerStepMean", "peakMemGB", "error")}, flush=True)
    print("elapsed min", round((time.time() - T0) / 60, 1), flush=True)


In [ ]:

# reward models (BT loss, 8 pairs/step at 512 tokens)
job("rm-smollm2-360m-full", "rm", "HuggingFaceTB/SmolLM2-360M", 300, mode="full", steps=10, pairs=8, seq=512)
job("rm-smollm2-360m-lora", "rm", "HuggingFaceTB/SmolLM2-360M", 300, mode="lora", steps=10, pairs=8, seq=512)
job("rm-q3-0.6b-lora", "rm", "Qwen/Qwen3-0.6B-Base", 300, mode="lora", steps=10, pairs=8, seq=512)
job("rm-q35-0.8b-lora", "rm", "Qwen/Qwen3.5-0.8B-Base", 360, mode="lora", steps=10, pairs=8, seq=512)
job("rm-q35-0.8b-full", "rm", "Qwen/Qwen3.5-0.8B-Base", 360, mode="full", steps=10, pairs=8, seq=512)
# DPO (manual loop, 4 pairs/step at <=512 tokens)
job("dpo-q3-0.6b-lora", "dpo", "Qwen/Qwen3-0.6B-Base", 300, mode="lora", steps=8, pairs=4, seq=512)
job("dpo-q35-0.8b-lora", "dpo", "Qwen/Qwen3.5-0.8B-Base", 360, mode="lora", steps=8, pairs=4, seq=512)
job("dpo-smollm2-360m-full", "dpo", "HuggingFaceTB/SmolLM2-360M", 300, mode="full", steps=8, pairs=4, seq=512)
job("dpo-q3-0.6b-full", "dpo", "Qwen/Qwen3-0.6B-Base", 300, mode="full", steps=8, pairs=4, seq=512)
# tool-use SFT: fp16 frozen base + fp32 LoRA, 4 x 768 per step
job("tool-q3-1.7b-lora16", "toolsft", "Qwen/Qwen3-1.7B-Base", 420, mode="lora16", steps=8, batch=4, micro=2, seq=768)
job("tool-smollm3-3b-lora16", "toolsft", "HuggingFaceTB/SmolLM3-3B-Base", 480, mode="lora16", steps=8, batch=4, micro=2, seq=768)
job("tool-q35-4b-lora16", "toolsft", "Qwen/Qwen3.5-4B-Base", 540, mode="lora16", steps=6, batch=4, micro=1, seq=768)
job("tool-q35-4b-qlora", "toolsft", "Qwen/Qwen3.5-4B-Base", 540, mode="qlora", steps=6, batch=4, micro=2, seq=768)
# TRL smoke
job("trl-rm-q3-0.6b", "trl_rm", "Qwen/Qwen3-0.6B-Base", 360, steps=6)
job("trl-dpo-q3-0.6b", "trl_dpo", "Qwen/Qwen3-0.6B-Base", 360, steps=6)
ALL["totalMinutes"] = round((time.time() - T0) / 60, 1); save(); print("DONE", ALL["totalMinutes"])
